# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution: Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) using the `mlcroissant` library. You will see how to inspect the dataset structure via its Croissant schema, extract record sets by their `@id`, and perform basic analysis.

### Dataset Source
Croissant schema URL: https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure 'mlcroissant' is installed
!pip install -q mlcroissant

## 1. Data Loading

Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset via mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"\nDataset Identifier: {metadata.identifier}")
print(f"Date Published: {metadata.datePublished}")
print(f"Authors: {', '.join([str(a) for a in getattr(metadata, 'author', [])])}")

## 2. Data Overview

Let's review the available record sets, fields, and their `@id`s within the dataset.

Below, we enumerate all record sets, their fields and columns, referencing entities by their `@id` per Croissant best practices.

In [ ]:
# List all record sets and corresponding field @ids
print("Record sets in this dataset:")
record_sets = []
for rs in dataset.record_sets:
    id_ = rs['@id']
    record_sets.append(id_)
    print(f"- RecordSet @id: {id_}")
    fields = rs.get('field', [])
    # Ensure 'field' is always a list
    if not isinstance(fields, list):
        fields = [fields]
    for f in fields:
        print(f"    - Field @id: {f.get('@id', f)}")
        if 'column' in f:
            columns = f['column']
            if not isinstance(columns, list):
                columns = [columns]
            for c in columns:
                print(f"        - Column @id: {c.get('@id', c)}")
if len(record_sets) == 0:
    print("No record sets found in croissant metadata. Attempting to enumerate via files...")
    for rec in dataset.distributions:
        print(f"- Distribution @id: {rec['@id']}")

## 3. Data Extraction

We will now extract tabular data from the primary record set. All access will be referenced using `@id` fields, as required.

Below, we attempt to detect and extract available record sets. For this demo, we'll focus on the first detected record set, or otherwise try common fallback Croissant identifiers.

In [ ]:
# Attempt to get all record set @ids
record_sets = [rs['@id'] for rs in dataset.record_sets]
# If none detected, try known fallback @id for this dataset
if len(record_sets) == 0:
    # Manually infer the @id for the main table per schema conventions
    # This is usually like 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/recordsets/primary' or similar
    # For this dataset, let's try the instance data after consulting schema structure:
    main_record_set_id = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/recordsets/second-primary-crc-records'
    record_sets = [main_record_set_id]
else:
    main_record_set_id = record_sets[0]

print(f"Using record set @id(s): {record_sets}")

dataframes = {}
for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records from: {record_set_id}")
    except Exception as e:
        print(f"Failed to load records from {record_set_id}: {e}")

# For demonstration, select the main dataframe
selected_rs_id = main_record_set_id
if selected_rs_id in dataframes:
    df = dataframes[selected_rs_id]
    print(f"Columns for record_set {selected_rs_id}:")
    print(list(df.columns))
    display(df.head(5))
else:
    print(f"No dataframe available for the main record set @id {selected_rs_id}")

## 4. Exploratory Data Analysis (EDA)

We will process and analyze the data by referencing all fields or columns using their `@id` (or schema alias). Common steps include filtering, normalization, and grouping.

In [ ]:
# --- EDA demo ---
df = dataframes[selected_rs_id]

# Identify a numeric field/column by inspecting the dataframe (users should replace this with correct @id)
print("List of columns for EDA (use their @id as needed):")
print(df.columns.tolist())

# For this dataset, let's assume there is a field representing 'age' and reference by its @id/column name
numeric_field_id = None
for c in df.columns:
    if 'age' in c.lower():
        numeric_field_id = c
        break
# If not found, use another numeric candidate
if not numeric_field_id:
    # Fallback: try first numeric-looking column
    for c in df.columns:
        if df[c].dtype in [np.int64, np.float64]:
            numeric_field_id = c
            break
if not numeric_field_id:
    print("No suitable numeric field found for analysis.")

# Filter for records with numeric_field > threshold
if numeric_field_id:
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with '{numeric_field_id}' > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalization
    col_norm = f"{numeric_field_id}_normalized"
    filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized '{numeric_field_id}':")
    print(filtered_df[[numeric_field_id, col_norm]].head())

    # Group by a key attribute (e.g., sex, tumor_location, etc.), reference by @id
    group_field = None
    for g in ['sex', 'gender', 'tumor_location', 'anatomical_location']:
        matches = [c for c in df.columns if g in c.lower()]
        if matches:
            group_field = matches[0]
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame()
        grouped_df = grouped_df.rename(columns={numeric_field_id: f"mean_{numeric_field_id} (per {group_field})"})
        print(f"\nGrouped mean of '{numeric_field_id}' by {group_field}:")
        print(grouped_df.head())
else:
    print("No numeric field available for EDA.")

## 5. Visualization

Let's create some simple visualizations to inspect data distributions and relationships, continuing to reference columns/fields via their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field (referenced by @id)
if numeric_field_id:
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

# Boxplot by group field if available
if numeric_field_id and group_field:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=df[group_field], y=df[numeric_field_id])
    plt.xlabel(group_field)
    plt.ylabel(numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.show()

## 6. Conclusion

- This notebook loaded the FAIR² dataset package using the Croissant schema URL and accessed all structures by their `@id`.
- We reviewed record sets, fields, and columns, and performed basic data analysis and visualization.
- For further analysis or modeling, always reference dataset fields and tables by their `@id` as per Croissant and `mlcroissant` best practices.